# AI NEWS — المذيع الافتراضي (MuseTalk 1.5 + Arabic TTS)

**هدف التجربة:** من صورة المذيع + النص العربي إلى فيديو 9:16 مع صوت مولّد تلقائيًا وحركة شفاه.

1. افتح Runtime → Change runtime type → **T4 GPU** (إذا توفر مجانًا).
2. اضغط **Runtime → Run all**.
3. إذا لم تكن الصورة مدمجة في الدفتر، ارفع **صورة واحدة فقط** عند ظهور نافذة الاختيار. **لا ترفع تسجيلًا صوتيًا.**
4. انتظر تثبيت MuseTalk وتنزيل النماذج (قد يستغرق 10–25 دقيقة أول مرة). ينزل MP4 تلقائيًا.

**الصوت:** Edge Neural Arabic TTS، صوت إماراتي ذكوري افتراضي. خدمة Edge TTS غير رسمية وقد تتوقف؛ ليست بديلًا إنتاجيًا مضمونًا عن Google Cloud TTS.

**الفيديو:** MuseTalk 1.5 مفتوح المصدر، يعمل على GPU في Colab. **Wav2Lip معطّل** بسبب قيود ترخيص الاستخدام التجاري.

**تنبيه:** تشغيل Colab يحتاج جلسة Google يفتحها صاحب الحساب؛ لا يمكن تشغيل GPU في حساب المستخدم من خلال GitHub فقط. لا يوجد فيديو مولّد حتى يتم تشغيل الدفتر.

**المصدر التقني:** [free-lipsync-colab](https://github.com/htunh/free-lipsync-colab) و [MuseTalk](https://github.com/TMElyralab/MuseTalk). استخدم فقط صورًا افتراضية أو صورًا بموافقة أصحابها، ووضّح للمشاهد أن المذيع مولّد بالذكاء الاصطناعي.


## Third-party notebook license

The MuseTalk Colab execution code is adapted from `htunh/free-lipsync-colab`.

MIT License — Copyright (c) 2026 free-lipsync-colab contributors.

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions: The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.


In [ ]:
# ⚙️ SETTINGS + STEP 1: GPU check
ENGINE = "musetalk"  #@param ["musetalk", "wav2lip", "both"]
#@markdown `musetalk` = MuseTalk 1.5 first, Wav2Lip only if it fails. `both` = make both. `wav2lip` = Wav2Lip only (non-commercial license!).
BBOX_SHIFT = 0  #@param {type:"slider", min:-9, max:9, step:1}
#@markdown Passed to MuseTalk `--bbox_shift` (mouth-region shift). Note: upstream MuseTalk **v1.5** currently forces this to 0 in `scripts/inference.py`, so it has effect only if the upstream code changes / v1 is used.
FPS = 25  #@param {type:"integer"}
OUTPUT_NAME = "AI_NEWS_Real_Estate_Anchor"  #@param {type:"string"}
import subprocess, os, re
OUTPUT_NAME = re.sub(r"[^A-Za-z0-9._-]+", "_", OUTPUT_NAME.strip()) or "lipsync"
OUTPUT_NAME = OUTPUT_NAME[:-4] if OUTPUT_NAME.lower().endswith(".mp4") else OUTPUT_NAME
FPS = int(FPS) if int(FPS) > 0 else 25
print("STEP 1/6: checking GPU...")
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    raise SystemExit("❌ No GPU found. Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.")
print("✅ GPU found:", r.stdout.strip())
for d in ("/content/work", "/content/output", "/content/inputs"): os.makedirs(d, exist_ok=True)
print(f"Settings: engine={ENGINE} bbox_shift={BBOX_SHIFT} fps={FPS} output={OUTPUT_NAME}.mp4")


In [ ]:
# STEP 2/6: AI NEWS — automatic Arabic voice + presenter image
# No recorded voice is required. TTS is generated directly from the script.
import os, subprocess, base64, urllib.request, shutil
TEXT = "بسم الله الرحمن الرحيم. نقدم لكم الآن نشرة الأخبار، وهي نشرة عقارية الأولى من نوعها في الوطن العربي."
VOICE = "ar-AE-HamdanNeural"  # Emirati male voice; try ar-SA-HamedNeural as an alternative
# Optional: a public HTTPS URL to the portrait image; otherwise choose a file once.
PORTRAIT_URL = ""
# For a fully prepared local copy of this notebook, an image can be embedded below.
EMBEDDED_FACE_B64 = ""
os.makedirs("/content/inputs", exist_ok=True)
FACE = "/content/inputs/ai-news-presenter.jpg"
if EMBEDDED_FACE_B64.strip():
    with open(FACE, "wb") as f: f.write(base64.b64decode(EMBEDDED_FACE_B64))
    print("✅ Presenter image loaded from this notebook")
elif PORTRAIT_URL.strip():
    if not PORTRAIT_URL.startswith("https://"): raise ValueError("PORTRAIT_URL must use HTTPS")
    urllib.request.urlretrieve(PORTRAIT_URL, FACE)
    print("✅ Presenter image downloaded")
else:
    print("📷 Upload ONLY the presenter's image. The Arabic voice is generated automatically.")
    from google.colab import files
    uploaded = files.upload()
    photos = [(name, data) for name, data in uploaded.items() if name.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
    if not photos: raise SystemExit("❌ Upload one presenter image (jpg/png/webp), then rerun this cell.")
    with open(FACE, "wb") as f: f.write(photos[0][1])
# Convert any uploaded image to a consistent 9:16 JPEG; face centered for lip detection.
from PIL import Image, ImageOps
im = Image.open(FACE).convert("RGB")
im = ImageOps.fit(im, (480, 854), method=Image.Resampling.LANCZOS, centering=(0.5,0.38))
im.save(FACE, quality=90)
print("🎙️ Generating Arabic voice from TEXT (no recording)...")
subprocess.run(["pip", "-q", "install", "edge-tts"], check=True)
AUDIO = "/content/work/ai_news_voice.mp3"
r = subprocess.run(["edge-tts", "--voice", VOICE, "--text", TEXT, "--write-media", AUDIO], capture_output=True, text=True)
if r.returncode != 0 or not os.path.exists(AUDIO) or os.path.getsize(AUDIO)<1000:
    raise RuntimeError("Arabic TTS failed: "+r.stderr[-600:])
FACE_IS_VIDEO = False
shutil.copy(FACE, "/content/work/face.jpg")
FACE = "/content/work/face.jpg"
print("✅ Portrait:", FACE, " | Audio:", AUDIO, " | Voice:", VOICE)


In [ ]:
# STEP 3/6: make a face video exactly as long as the audio
import subprocess, json
print("STEP 3/6: preparing the face video...")
def sh(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-3000:], p.stderr[-3000:]); raise SystemExit("❌ Command failed: " + cmd[:200])
    return p.stdout
DUR = float(sh(f'ffprobe -v error -show_entries format=duration -of csv=p=0 "{AUDIO}"').strip())
info = json.loads(sh(f'ffprobe -v error -select_streams v:0 -show_entries stream=width,height -of json "{FACE}"'))["streams"][0]
W, H = info["width"] // 2 * 2, info["height"] // 2 * 2
FRAMES = int(round(DUR * FPS)) + 2
if FACE_IS_VIDEO:
    # loop the clip if shorter than the audio, trim to length, constant fps
    sh(f'ffmpeg -y -v error -stream_loop -1 -i "{FACE}" -an -t {DUR + 0.2:.3f} -vf "scale={W}:{H},fps={FPS},format=yuv420p" '
       f'-c:v libx264 -crf 15 -preset fast /content/work/still.mp4')
else:
    zoom_step = 0.05 / max(FRAMES, 1)   # gentle 1.00 -> 1.05 zoom so a still photo isn't frozen
    vf = (f"scale={W*2}:{H*2}:flags=lanczos,"
          f"zoompan=z='1+{zoom_step:.8f}*on':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={FRAMES}:s={W}x{H}:fps={FPS},format=yuv420p")
    sh(f'ffmpeg -y -v error -i "{FACE}" -vf "{vf}" -frames:v {FRAMES} -c:v libx264 -crf 15 -preset fast /content/work/still.mp4')
sh(f'ffmpeg -y -v error -i "{AUDIO}" -ac 1 -ar 16000 /content/work/voice16k.wav')
print(f"✅ Face video: {W}x{H}, {FPS} fps, {DUR:.1f} s")


In [ ]:
# STEP 4/6: MuseTalk 1.5 – separate pinned Python 3.10 env (uv), public weights

# ---------- MuseTalk helpers (same code in the RETRY cell) ----------
import subprocess, os, textwrap, sys, time, glob
MT_LOG = "/content/work/musetalk.log"
PY = "/content/mtenv/bin/python"
MT_DIR = "/content/MuseTalk"
def mt_run(cmd, label):
    """Run a command, stream it to musetalk.log, return (ok, output)."""
    print(f"   ▶ {label}")
    with open(MT_LOG, "a") as log:
        log.write(f"\n\n===== {time.strftime('%H:%M:%S')} {label}\n$ {cmd}\n")
    p = subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True)
    out = (p.stdout or "") + (p.stderr or "")
    with open(MT_LOG, "a") as log:
        log.write(out + f"\n[exit code {p.returncode}]\n")
    return p.returncode == 0, out
def mt_fail(label, out):
    lines = out.strip().splitlines()[-60:]
    print("\n" + "=" * 70)
    print(f"❌ MuseTalk FAILED at: {label}")
    print("---- last 60 lines of output (screenshot this box) ----")
    print("\n".join(lines) if lines else "(no output)")
    print("=" * 70)
    print("Full log saved to", MT_LOG, "(📁 folder icon > work > musetalk.log)")
    mt_env_report()
    raise RuntimeError(label)
def mt_env_report():
    if not os.path.exists(PY):
        print("ENV: MuseTalk Python env not installed"); return
    chk = r"""
import sys, importlib
print('python', sys.version.split()[0])
try:
    import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available(), '| cuda', torch.version.cuda)
except Exception as e: print('torch import FAILED:', repr(e))
for m in ['numpy','cv2','mmcv','mmengine','mmdet','mmpose','mmpose.apis','mmcv.ops','diffusers','transformers','huggingface_hub','librosa','xtcocotools']:
    try:
        x = importlib.import_module(m); print(f'{m:16s} OK ', getattr(x,'__version__',''))
    except Exception as e: print(f'{m:16s} FAILED:', repr(e)[:200])
"""
    p = subprocess.run([PY, "-c", chk], capture_output=True, text=True, cwd=MT_DIR if os.path.isdir(MT_DIR) else None)
    print("---- environment check ----"); print(p.stdout, p.stderr[-1500:])
    with open(MT_LOG, "a") as log: log.write("\n===== ENV CHECK\n" + p.stdout + p.stderr)

def mt_install():
    PIP = f"{PY} -m pip install -q"
    steps = []
    if not os.path.isdir(MT_DIR):
        steps.append(("clone MuseTalk", f"git clone -q https://github.com/TMElyralab/MuseTalk {MT_DIR}"))
    if not os.path.exists(PY):
        steps += [("install uv", "pip -q install uv"),
                  ("create Python 3.10 env", "uv venv -q --seed --python 3.10 /content/mtenv")]
    marker = "/content/mtenv/.musetalk_ok"
    if not os.path.exists(marker):
        steps += [
          ("torch 2.0.1 cu118", f"{PIP} torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118"),
          ("MuseTalk requirements (pinned)", f"{PIP} numpy==1.23.5 diffusers==0.30.2 accelerate==0.28.0 transformers==4.39.2 tokenizers==0.15.2 huggingface_hub==0.30.2 "
             "opencv-python==4.9.0.80 soundfile==0.12.1 librosa==0.11.0 numba==0.59.1 einops==0.8.1 omegaconf==2.3.0 "
             "ffmpeg-python==0.2.0 moviepy==1.0.3 'imageio[ffmpeg]==2.34.0' tqdm requests"),
          ("mmengine", f"{PIP} mmengine==0.10.4"),
          ("mmcv 2.0.1 prebuilt wheel", f"{PIP} --only-binary=mmcv mmcv==2.0.1 -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html"),
          ("mmdet 3.1.0", f"{PIP} mmdet==3.1.0"),
          ("mmpose 1.1.0 (no chumpy)", f"{PIP} json_tricks munkres xtcocotools==1.14.3 'scipy==1.11.4' 'matplotlib==3.8.4' 'pycocotools==2.0.7' 'shapely==2.0.4' terminaltables && {PIP} --no-deps mmpose==1.1.0"),
          # anything above may have dragged numpy/opencv up -> force the pinned ones back
          # mmengine imports pkg_resources; uv-made envs may lack it -> pin setuptools<70 (still ships pkg_resources)
          ("setuptools<70 (pkg_resources)", f"{PIP} 'setuptools<70'"),
          ("re-pin numpy/opencv", f"{PIP} --force-reinstall --no-deps numpy==1.23.5 opencv-python==4.9.0.80"),
        ]
    for label, cmd in steps:
        ok, out = mt_run(cmd, label)
        if not ok: mt_fail(label, out)
    ok, out = mt_run(f"{PY} -c 'import pkg_resources' || {PIP} 'setuptools<70'", "ensure pkg_resources")
    if not ok: mt_fail("ensure pkg_resources", out)
    # import exactly what inference.py imports, from the repo folder
    ok, out = mt_run(f"cd {MT_DIR} && {PY} -c \"import pkg_resources,torch,cv2,numpy,mmcv,mmcv.ops,mmdet,mmpose,mmpose.apis; from mmpose.apis import init_model; "
                     "from transformers import WhisperModel; from diffusers import AutoencoderKL; import musetalk.utils.utils; "
                     "print('imports OK | torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| numpy', numpy.__version__)\"",
                     "import check")
    if not ok or "cuda True" not in out: mt_fail("import check (cuda must be True)", out)
    print("   ", out.strip().splitlines()[-1])
    open(marker, "w").write("ok")

def mt_weights():
    HFR = "https://huggingface.co"; M = f"{MT_DIR}/models"
    dl = {
        f"{M}/musetalkV15/unet.pth":               f"{HFR}/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth",
        f"{M}/musetalkV15/musetalk.json":          f"{HFR}/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json",
        f"{M}/sd-vae/config.json":                 f"{HFR}/stabilityai/sd-vae-ft-mse/resolve/main/config.json",
        f"{M}/sd-vae/diffusion_pytorch_model.bin": f"{HFR}/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin",
        f"{M}/whisper/config.json":                f"{HFR}/openai/whisper-tiny/resolve/main/config.json",
        f"{M}/whisper/pytorch_model.bin":          f"{HFR}/openai/whisper-tiny/resolve/main/pytorch_model.bin",
        f"{M}/whisper/preprocessor_config.json":   f"{HFR}/openai/whisper-tiny/resolve/main/preprocessor_config.json",
        f"{M}/dwpose/dw-ll_ucoco_384.pth":         f"{HFR}/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth",
        f"{M}/face-parse-bisent/79999_iter.pth":   f"{HFR}/ManyOtherFunctions/face-parse-bisent/resolve/main/79999_iter.pth",
        f"{M}/face-parse-bisent/resnet18-5c106cde.pth": "https://download.pytorch.org/models/resnet18-5c106cde.pth",
        # face detector: otherwise downloaded at run time from adrianbulat.com (often slow/down)
        f"{MT_DIR}/musetalk/utils/face_detection/detection/sfd/s3fd.pth": f"{HFR}/camenduru/Wav2Lip/resolve/main/face_detection/detection/sfd/s3fd.pth",
    }
    for path, url in dl.items():
        if os.path.exists(path) and os.path.getsize(path) > 1000: continue
        os.makedirs(os.path.dirname(path), exist_ok=True)
        ok, out = mt_run(f'wget -q -O "{path}.part" "{url}" && mv "{path}.part" "{path}"', "download " + os.path.basename(path))
        if not ok: mt_fail("download " + url, out)
    print("   ✅ all MuseTalk models present")

def mt_patch_upstream():
    """Upstream scripts/inference.py skips (continue) any frame whose face bbox is missing/degenerate when
    writing result PNGs. That leaves gaps (e.g. no 00000000.png) and its ffmpeg image2 step then fails with
    'Could find no file ... %08d.png and index in the range 0-4'. Patch: write the original frame instead,
    so the PNG sequence is always complete. Idempotent."""
    p = f"{MT_DIR}/scripts/inference.py"
    s = open(p).read()
    if "LIPSYNC_COLAB_PATCH" in s: return
    old = """                try:
                    res_frame = cv2.resize(res_frame.astype(np.uint8), (x2-x1, y2-y1))
                except:
                    continue
"""
    new = """                try:  # LIPSYNC_COLAB_PATCH: keep the PNG sequence gap-free
                    if bbox == coord_placeholder or x2 <= x1 or y2 <= y1: raise ValueError("no face")
                    res_frame = cv2.resize(res_frame.astype(np.uint8), (x2-x1, y2-y1))
                except Exception:
                    cv2.imwrite(f"{result_img_save_path}/{str(i).zfill(8)}.png", ori_frame)
                    continue
"""
    if old not in s:
        print("   ⚠️ upstream inference.py changed; frame-gap patch not applied"); return
    open(p, "w").write(s.replace(old, new)); print("   ▶ patched upstream inference.py (gap-free frames)")

def mt_find_output(out_dir, expected):
    if os.path.exists(expected) and os.path.getsize(expected) > 1000: return expected
    c = [f for f in glob.glob(f"{out_dir}/**/*.mp4", recursive=True)
         if not os.path.basename(f).startswith("temp_") and not f.endswith("_concat.mp4") and os.path.getsize(f) > 1000]
    return max(c, key=os.path.getmtime) if c else None

def mt_infer(video, audio):
    FPS = globals().get("FPS", 25); BBOX_SHIFT = globals().get("BBOX_SHIFT", 0)
    out_dir = "/content/work/mt"; out_mp4 = f"{out_dir}/v15/musetalk_raw.mp4"
    subprocess.run(f"rm -rf {out_dir}", shell=True)
    with open("/content/work/mt.yaml", "w") as f:
        f.write(f'task_0:\n  video_path: "{video}"\n  audio_path: "{audio}"\n')
    # identical to the repo's inference.sh for "v1.5 normal", with absolute paths + fp16 for T4
    cmd = (f"cd {MT_DIR} && PYTHONUNBUFFERED=1 {PY} -m scripts.inference "
           f"--inference_config /content/work/mt.yaml --result_dir {out_dir} "
           f"--unet_model_path ./models/musetalkV15/unet.pth --unet_config ./models/musetalkV15/musetalk.json "
           f"--version v15 --whisper_dir ./models/whisper --vae_type sd-vae "
           f"--ffmpeg_path /usr/bin --fps {FPS} --batch_size 8 --bbox_shift {BBOX_SHIFT} --use_float16 --output_vid_name musetalk_raw.mp4")
    mt_patch_upstream()
    ok, out = mt_run(cmd, "MuseTalk inference")
    found = mt_find_output(out_dir, out_mp4)
    # inference.py catches errors inside each task and still exits 0, so check the output file and the log text
    if not found:
        mt_fail("MuseTalk inference (no output video; if the log says no face was found, use a clearer frontal face)", out)
    if "Error occurred during processing" in out:
        print("   ⚠️ MuseTalk reported an error after writing a video; using", found)
    return found

def musetalk_all(video, audio):
    try:
        print("STEP 4/6: MuseTalk – installing (5-8 min first time, skipped if already installed)...")
        mt_install()
        print("STEP 4/6: MuseTalk – downloading models (~3.5 GB, skipped if present)...")
        mt_weights()
        print("STEP 4/6: MuseTalk – lip-syncing (about 2-6 min)...")
        return mt_infer(video, audio)
    except RuntimeError:
        return None

MT_OUT = None
if ENGINE in ("musetalk", "both"):
    MT_OUT = musetalk_all("/content/work/still.mp4", "/content/work/voice16k.wav")
    print("✅ MuseTalk done" if MT_OUT else "⚠️ MuseTalk didn't work – Wav2Lip fallback disabled (commercial licensing).")
else:
    print("STEP 4/6: MuseTalk skipped (ENGINE setting)")
MT_OK = bool(MT_OUT)


In [ ]:
# STEP 5/6: Wav2Lip intentionally disabled for AI NEWS (commercial news service)
W2L_OK = False
print("STEP 5/6: Wav2Lip disabled. MuseTalk only.")


In [ ]:
# STEP 6/6: mux original audio back in at full quality and download
import subprocess, os
from google.colab import files
print("STEP 6/6: finishing the video (original voice, AAC 192k, H.264)...")
def finish(src, dst):
    cmd = (f'ffmpeg -y -v error -i "{src}" -i "{AUDIO}" -map 0:v:0 -map 1:a:0 '
           f'-vf "scale={W}:{H}:flags=lanczos,format=yuv420p" -r {FPS} -c:v libx264 -preset medium -crf 17 '
           f'-c:a aac -b:a 192k -shortest -movflags +faststart "{dst}"')
    return subprocess.run(cmd, shell=True).returncode == 0
outs = []
if W2L_OK and finish("/content/work/wav2lip_raw.mp4", f"/content/output/{OUTPUT_NAME}-wav2lip.mp4"):
    outs.append(f"/content/output/{OUTPUT_NAME}-wav2lip.mp4")
if MT_OK and finish(MT_OUT, f"/content/output/{OUTPUT_NAME}.mp4"):
    outs.append(f"/content/output/{OUTPUT_NAME}.mp4")
elif W2L_OK:
    subprocess.run(f"cp /content/output/{OUTPUT_NAME}-wav2lip.mp4 /content/output/{OUTPUT_NAME}.mp4", shell=True)
    outs.append(f"/content/output/{OUTPUT_NAME}.mp4")
if not outs:
    raise SystemExit("❌ MuseTalk failed. See the ❌ messages above (and /content/work/musetalk.log).")
for o in dict.fromkeys(outs):
    print("⬇️ downloading", os.path.basename(o), f"({os.path.getsize(o)/1e6:.1f} MB)")
    try: files.download(o)
    except Exception as e: print('   (auto-download failed – use the DOWNLOAD cell or the 📁 Files panel)')
print(f"🎉 ALL DONE. Engine used for {OUTPUT_NAME}.mp4:", "MuseTalk 1.5" if MT_OK else "Wav2Lip")
print("If nothing downloaded: click the 📁 folder icon on the left > output > right-click the file > Download.")

---
## 🔁 RETRY MUSETALK ONLY
Run this cell by itself (▶ / Shift+Enter) after fixing something, e.g. if MuseTalk failed. **Don't** use Run all.
It reuses your uploaded files and anything already installed (~3–6 min). If the runtime was reset, use Runtime → Run all instead.
You can change BBOX_SHIFT in the Settings cell, rerun that cell, then rerun this one.

In [ ]:
# RETRY MUSETALK ONLY

# ---------- MuseTalk helpers (same code in the RETRY cell) ----------
import subprocess, os, textwrap, sys, time, glob
MT_LOG = "/content/work/musetalk.log"
PY = "/content/mtenv/bin/python"
MT_DIR = "/content/MuseTalk"
def mt_run(cmd, label):
    """Run a command, stream it to musetalk.log, return (ok, output)."""
    print(f"   ▶ {label}")
    with open(MT_LOG, "a") as log:
        log.write(f"\n\n===== {time.strftime('%H:%M:%S')} {label}\n$ {cmd}\n")
    p = subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True)
    out = (p.stdout or "") + (p.stderr or "")
    with open(MT_LOG, "a") as log:
        log.write(out + f"\n[exit code {p.returncode}]\n")
    return p.returncode == 0, out
def mt_fail(label, out):
    lines = out.strip().splitlines()[-60:]
    print("\n" + "=" * 70)
    print(f"❌ MuseTalk FAILED at: {label}")
    print("---- last 60 lines of output (screenshot this box) ----")
    print("\n".join(lines) if lines else "(no output)")
    print("=" * 70)
    print("Full log saved to", MT_LOG, "(📁 folder icon > work > musetalk.log)")
    mt_env_report()
    raise RuntimeError(label)
def mt_env_report():
    if not os.path.exists(PY):
        print("ENV: MuseTalk Python env not installed"); return
    chk = r"""
import sys, importlib
print('python', sys.version.split()[0])
try:
    import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available(), '| cuda', torch.version.cuda)
except Exception as e: print('torch import FAILED:', repr(e))
for m in ['numpy','cv2','mmcv','mmengine','mmdet','mmpose','mmpose.apis','mmcv.ops','diffusers','transformers','huggingface_hub','librosa','xtcocotools']:
    try:
        x = importlib.import_module(m); print(f'{m:16s} OK ', getattr(x,'__version__',''))
    except Exception as e: print(f'{m:16s} FAILED:', repr(e)[:200])
"""
    p = subprocess.run([PY, "-c", chk], capture_output=True, text=True, cwd=MT_DIR if os.path.isdir(MT_DIR) else None)
    print("---- environment check ----"); print(p.stdout, p.stderr[-1500:])
    with open(MT_LOG, "a") as log: log.write("\n===== ENV CHECK\n" + p.stdout + p.stderr)

def mt_install():
    PIP = f"{PY} -m pip install -q"
    steps = []
    if not os.path.isdir(MT_DIR):
        steps.append(("clone MuseTalk", f"git clone -q https://github.com/TMElyralab/MuseTalk {MT_DIR}"))
    if not os.path.exists(PY):
        steps += [("install uv", "pip -q install uv"),
                  ("create Python 3.10 env", "uv venv -q --seed --python 3.10 /content/mtenv")]
    marker = "/content/mtenv/.musetalk_ok"
    if not os.path.exists(marker):
        steps += [
          ("torch 2.0.1 cu118", f"{PIP} torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118"),
          ("MuseTalk requirements (pinned)", f"{PIP} numpy==1.23.5 diffusers==0.30.2 accelerate==0.28.0 transformers==4.39.2 tokenizers==0.15.2 huggingface_hub==0.30.2 "
             "opencv-python==4.9.0.80 soundfile==0.12.1 librosa==0.11.0 numba==0.59.1 einops==0.8.1 omegaconf==2.3.0 "
             "ffmpeg-python==0.2.0 moviepy==1.0.3 'imageio[ffmpeg]==2.34.0' tqdm requests"),
          ("mmengine", f"{PIP} mmengine==0.10.4"),
          ("mmcv 2.0.1 prebuilt wheel", f"{PIP} --only-binary=mmcv mmcv==2.0.1 -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html"),
          ("mmdet 3.1.0", f"{PIP} mmdet==3.1.0"),
          ("mmpose 1.1.0 (no chumpy)", f"{PIP} json_tricks munkres xtcocotools==1.14.3 'scipy==1.11.4' 'matplotlib==3.8.4' 'pycocotools==2.0.7' 'shapely==2.0.4' terminaltables && {PIP} --no-deps mmpose==1.1.0"),
          # anything above may have dragged numpy/opencv up -> force the pinned ones back
          # mmengine imports pkg_resources; uv-made envs may lack it -> pin setuptools<70 (still ships pkg_resources)
          ("setuptools<70 (pkg_resources)", f"{PIP} 'setuptools<70'"),
          ("re-pin numpy/opencv", f"{PIP} --force-reinstall --no-deps numpy==1.23.5 opencv-python==4.9.0.80"),
        ]
    for label, cmd in steps:
        ok, out = mt_run(cmd, label)
        if not ok: mt_fail(label, out)
    ok, out = mt_run(f"{PY} -c 'import pkg_resources' || {PIP} 'setuptools<70'", "ensure pkg_resources")
    if not ok: mt_fail("ensure pkg_resources", out)
    # import exactly what inference.py imports, from the repo folder
    ok, out = mt_run(f"cd {MT_DIR} && {PY} -c \"import pkg_resources,torch,cv2,numpy,mmcv,mmcv.ops,mmdet,mmpose,mmpose.apis; from mmpose.apis import init_model; "
                     "from transformers import WhisperModel; from diffusers import AutoencoderKL; import musetalk.utils.utils; "
                     "print('imports OK | torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| numpy', numpy.__version__)\"",
                     "import check")
    if not ok or "cuda True" not in out: mt_fail("import check (cuda must be True)", out)
    print("   ", out.strip().splitlines()[-1])
    open(marker, "w").write("ok")

def mt_weights():
    HFR = "https://huggingface.co"; M = f"{MT_DIR}/models"
    dl = {
        f"{M}/musetalkV15/unet.pth":               f"{HFR}/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth",
        f"{M}/musetalkV15/musetalk.json":          f"{HFR}/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json",
        f"{M}/sd-vae/config.json":                 f"{HFR}/stabilityai/sd-vae-ft-mse/resolve/main/config.json",
        f"{M}/sd-vae/diffusion_pytorch_model.bin": f"{HFR}/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin",
        f"{M}/whisper/config.json":                f"{HFR}/openai/whisper-tiny/resolve/main/config.json",
        f"{M}/whisper/pytorch_model.bin":          f"{HFR}/openai/whisper-tiny/resolve/main/pytorch_model.bin",
        f"{M}/whisper/preprocessor_config.json":   f"{HFR}/openai/whisper-tiny/resolve/main/preprocessor_config.json",
        f"{M}/dwpose/dw-ll_ucoco_384.pth":         f"{HFR}/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth",
        f"{M}/face-parse-bisent/79999_iter.pth":   f"{HFR}/ManyOtherFunctions/face-parse-bisent/resolve/main/79999_iter.pth",
        f"{M}/face-parse-bisent/resnet18-5c106cde.pth": "https://download.pytorch.org/models/resnet18-5c106cde.pth",
        # face detector: otherwise downloaded at run time from adrianbulat.com (often slow/down)
        f"{MT_DIR}/musetalk/utils/face_detection/detection/sfd/s3fd.pth": f"{HFR}/camenduru/Wav2Lip/resolve/main/face_detection/detection/sfd/s3fd.pth",
    }
    for path, url in dl.items():
        if os.path.exists(path) and os.path.getsize(path) > 1000: continue
        os.makedirs(os.path.dirname(path), exist_ok=True)
        ok, out = mt_run(f'wget -q -O "{path}.part" "{url}" && mv "{path}.part" "{path}"', "download " + os.path.basename(path))
        if not ok: mt_fail("download " + url, out)
    print("   ✅ all MuseTalk models present")

def mt_patch_upstream():
    """Upstream scripts/inference.py skips (continue) any frame whose face bbox is missing/degenerate when
    writing result PNGs. That leaves gaps (e.g. no 00000000.png) and its ffmpeg image2 step then fails with
    'Could find no file ... %08d.png and index in the range 0-4'. Patch: write the original frame instead,
    so the PNG sequence is always complete. Idempotent."""
    p = f"{MT_DIR}/scripts/inference.py"
    s = open(p).read()
    if "LIPSYNC_COLAB_PATCH" in s: return
    old = """                try:
                    res_frame = cv2.resize(res_frame.astype(np.uint8), (x2-x1, y2-y1))
                except:
                    continue
"""
    new = """                try:  # LIPSYNC_COLAB_PATCH: keep the PNG sequence gap-free
                    if bbox == coord_placeholder or x2 <= x1 or y2 <= y1: raise ValueError("no face")
                    res_frame = cv2.resize(res_frame.astype(np.uint8), (x2-x1, y2-y1))
                except Exception:
                    cv2.imwrite(f"{result_img_save_path}/{str(i).zfill(8)}.png", ori_frame)
                    continue
"""
    if old not in s:
        print("   ⚠️ upstream inference.py changed; frame-gap patch not applied"); return
    open(p, "w").write(s.replace(old, new)); print("   ▶ patched upstream inference.py (gap-free frames)")

def mt_find_output(out_dir, expected):
    if os.path.exists(expected) and os.path.getsize(expected) > 1000: return expected
    c = [f for f in glob.glob(f"{out_dir}/**/*.mp4", recursive=True)
         if not os.path.basename(f).startswith("temp_") and not f.endswith("_concat.mp4") and os.path.getsize(f) > 1000]
    return max(c, key=os.path.getmtime) if c else None

def mt_infer(video, audio):
    FPS = globals().get("FPS", 25); BBOX_SHIFT = globals().get("BBOX_SHIFT", 0)
    out_dir = "/content/work/mt"; out_mp4 = f"{out_dir}/v15/musetalk_raw.mp4"
    subprocess.run(f"rm -rf {out_dir}", shell=True)
    with open("/content/work/mt.yaml", "w") as f:
        f.write(f'task_0:\n  video_path: "{video}"\n  audio_path: "{audio}"\n')
    # identical to the repo's inference.sh for "v1.5 normal", with absolute paths + fp16 for T4
    cmd = (f"cd {MT_DIR} && PYTHONUNBUFFERED=1 {PY} -m scripts.inference "
           f"--inference_config /content/work/mt.yaml --result_dir {out_dir} "
           f"--unet_model_path ./models/musetalkV15/unet.pth --unet_config ./models/musetalkV15/musetalk.json "
           f"--version v15 --whisper_dir ./models/whisper --vae_type sd-vae "
           f"--ffmpeg_path /usr/bin --fps {FPS} --batch_size 8 --bbox_shift {BBOX_SHIFT} --use_float16 --output_vid_name musetalk_raw.mp4")
    mt_patch_upstream()
    ok, out = mt_run(cmd, "MuseTalk inference")
    found = mt_find_output(out_dir, out_mp4)
    # inference.py catches errors inside each task and still exits 0, so check the output file and the log text
    if not found:
        mt_fail("MuseTalk inference (no output video; if the log says no face was found, use a clearer frontal face)", out)
    if "Error occurred during processing" in out:
        print("   ⚠️ MuseTalk reported an error after writing a video; using", found)
    return found

def musetalk_all(video, audio):
    try:
        print("STEP 4/6: MuseTalk – installing (5-8 min first time, skipped if already installed)...")
        mt_install()
        print("STEP 4/6: MuseTalk – downloading models (~3.5 GB, skipped if present)...")
        mt_weights()
        print("STEP 4/6: MuseTalk – lip-syncing (about 2-6 min)...")
        return mt_infer(video, audio)
    except RuntimeError:
        return None

import glob, json
from google.colab import files
print("RETRY: checking what is already here...")
if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
    raise SystemExit("❌ No GPU. Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")
FPS = globals().get("FPS", 25); OUTPUT_NAME = globals().get("OUTPUT_NAME", "lipsync")
aud = glob.glob("/content/work/voice.*")
if not aud or not os.path.exists("/content/work/still.mp4"):
    raise SystemExit("❌ Your input files are gone (runtime was reset). Use Runtime > Run all instead.")
AUDIO = aud[0]
if not os.path.exists("/content/work/voice16k.wav"):
    subprocess.run(f'ffmpeg -y -v error -i "{AUDIO}" -ac 1 -ar 16000 /content/work/voice16k.wav', shell=True, check=True)
st = json.loads(subprocess.run('ffprobe -v error -select_streams v:0 -show_entries stream=width,height -of json /content/work/still.mp4',
                               shell=True, capture_output=True, text=True).stdout)["streams"][0]
W, H = st["width"], st["height"]
open(MT_LOG, "a").write("\n\n######## RETRY " + time.strftime("%H:%M:%S") + "\n")
MT_OUT = musetalk_all("/content/work/still.mp4", "/content/work/voice16k.wav")
if not MT_OUT:
    raise SystemExit("❌ MuseTalk still failed. See the error box above and /content/work/musetalk.log.")
MT_OK = True
os.makedirs("/content/output", exist_ok=True)
dst = f"/content/output/{OUTPUT_NAME}-musetalk.mp4"
subprocess.run(f'ffmpeg -y -v error -i "{MT_OUT}" -i "{AUDIO}" -map 0:v:0 -map 1:a:0 -vf "scale={W}:{H}:flags=lanczos,format=yuv420p" '
               f'-r {FPS} -c:v libx264 -preset medium -crf 17 -c:a aac -b:a 192k -shortest -movflags +faststart "{dst}"', shell=True, check=True)
print(f"⬇️ downloading {os.path.basename(dst)} ({os.path.getsize(dst)/1e6:.1f} MB)")
try: files.download(dst)
except Exception: print("   (auto-download failed – use the DOWNLOAD cell or the 📁 Files panel)")
print("🎉 DONE (MuseTalk).")


---
## ⬇️ DOWNLOAD RESULTS
Re-download everything in `/content/output` (zipped if more than one file). Or use 📁 Files → `output` → right-click → Download.

In [ ]:
# DOWNLOAD all results
import glob, os, subprocess
from google.colab import files
outs = sorted(glob.glob("/content/output/*.mp4"))
if not outs: raise SystemExit("❌ Nothing in /content/output yet. Run the steps above first.")
for o in outs: print(f"  {os.path.basename(o)}  {os.path.getsize(o)/1e6:.1f} MB")
if len(outs) == 1: files.download(outs[0])
else:
    subprocess.run("cd /content/output && rm -f results.zip && zip -q results.zip *.mp4", shell=True, check=True)
    files.download("/content/output/results.zip")


---
## 🩺 DIAGNOSTICS (optional)
Prints the MuseTalk environment check and the last 60 lines of `/content/work/musetalk.log`. Include this output in bug reports.

In [ ]:
# DIAGNOSTICS
import os, subprocess
PY = "/content/mtenv/bin/python"; LOG = "/content/work/musetalk.log"
print(subprocess.run("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
if os.path.exists(PY):
    chk = "import importlib,sys;print('python',sys.version.split()[0])\nfor m in ['pkg_resources','torch','numpy','cv2','mmcv','mmengine','mmdet','mmpose','mmpose.apis','mmcv.ops','diffusers','transformers']:\n  try: x=importlib.import_module(m);print(f'{m:15s} OK',getattr(x,'__version__',''))\n  except Exception as e: print(f'{m:15s} FAILED',repr(e)[:200])"
    print(subprocess.run([PY, "-c", chk], capture_output=True, text=True, cwd="/content/MuseTalk" if os.path.isdir("/content/MuseTalk") else None).stdout)
else: print("MuseTalk env not installed (/content/mtenv missing)")
if os.path.exists(LOG): print("---- last 60 lines of", LOG, "----\n" + "\n".join(open(LOG, errors="replace").read().splitlines()[-60:]))
